# HyPE Implementation
This notebook implements the Hypothetical Prompt Embeddings (HyPE) method as described in the paper for Retrieval-Augmented Generation (RAG).

In [ ]:
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from tqdm import tqdm

In [ ]:
# Load embedding model
embedding_model = SentenceTransformer('BAAI/bge-m3')

# Load question generation pipeline
question_generator = pipeline("text2text-generation", model="mrm8488/t5-base-finetuned-question-generation-ap")

In [ ]:
# Sample corpus
corpus = [
    "Retrieval-Augmented Generation (RAG) enhances language models with document retrieval capabilities.",
    "RAG allows factual information to be integrated into generative responses using external sources.",
    "HyPE generates hypothetical questions during indexing instead of query time to optimize retrieval."
]

# Chunking - simplified as direct usage of sentences
chunks = corpus

In [ ]:
# Generate multiple hypothetical questions per chunk
def generate_hypothetical_prompts(text, num_prompts=3):
    prompts = []
    for _ in range(num_prompts):
        result = question_generator(text, max_length=64, do_sample=True, top_k=50, top_p=0.95)[0]['generated_text']
        prompts.append(result)
    return prompts

chunk_prompts = {i: generate_hypothetical_prompts(chunk) for i, chunk in enumerate(chunks)}
chunk_prompts

In [ ]:
# Create an index of (vector, chunk) pairs
index = []
for chunk_id, prompts in chunk_prompts.items():
    for prompt in prompts:
        vector = embedding_model.encode(prompt, convert_to_tensor=True)
        index.append((vector, chunks[chunk_id]))

In [ ]:
# Retrieval function
def retrieve(query, top_k=3):
    query_vec = embedding_model.encode(query, convert_to_tensor=True)
    results = sorted(index, key=lambda x: util.cos_sim(query_vec, x[0]), reverse=True)
    return [chunk for _, chunk in results[:top_k]]

# Example query
retrieve("How does HyPE improve RAG retrieval?")